**Part 1 · AI Fundamentals** · step 2 of 25  
[← 01 How LLMs work](01_how_llms_work.ipynb) · [🏠 Roadmap](../README.md) · [03 Python prerequisites →](03_python_prerequisites.ipynb)

# 02 · From LLMs to Agents — the big picture

**What you'll learn**
- The "ladder" of LLM applications, from one call to multi-agent graphs
- **Tool calling** — how a model "uses" a function (hands-on with the raw SDK)
- What an **agent** is (the ReAct loop) and how it differs from a **workflow**
- The ecosystem: **LangChain**, **LangGraph**, **LangSmith** — and what each is for

## 1. The ladder of LLM applications

| Level | Pattern | Who decides the steps? | Example | In this course |
|---|---|---|---|---|
| 1 | **Single call** | you | summarise this text | 01, 04–05 |
| 2 | **Chain / workflow** | you (fixed order) | prompt template → model → parser | 06 |
| 3 | **Tool calling** | model picks a tool, you run it | "weather in Delhi?" → `get_weather` | 09 |
| 4 | **Agent** | model, in a loop, until done | research assistant, SQL assistant | 10–15, 21 |
| 5 | **Graph / multi-step system** | you design the graph, model decides inside nodes | custom routing, approvals, multi-agent | 16–19 |

Climb only as high as you need: lower levels are cheaper, faster and more predictable.

## 2. Tool calling — how a model "uses" a function

The model **cannot run code**. Instead:
1. You describe your functions with a **JSON schema** (name, description, parameters).
2. The model replies with a **tool call**: *"call `get_weather` with `{"city": "Delhi"}`"*.
3. **Your code** runs the function and sends the result back as a `tool` message.
4. The model uses the result to write the final answer.

Let's do it with the raw OpenAI SDK.

In [1]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

True

In [2]:
import json
from openai import OpenAI

client = OpenAI()


def get_weather(city: str) -> str:
    """Pretend weather API — a real one would make an HTTP request."""
    return f"It is sunny and 30°C in {city}"


# The schema the model sees. Writing this by hand for every function is tedious...
TOOLS = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name"}},
            "required": ["city"],
        },
    },
}]

In [3]:
# Step 1-2: the model decides to call a tool
messages = [{"role": "user", "content": "What's the weather in Delhi?"}]
first = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=TOOLS)
assistant_msg = first.choices[0].message

print("content   :", assistant_msg.content)  # None — it wants a tool instead
print("tool_calls:", assistant_msg.tool_calls)

content   : None
tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_Lcy8yUxhhrwQ06khTD2ls4RF', function=Function(arguments='{"city":"Delhi"}', name='get_weather'), type='function')]


In [4]:
# Step 3: OUR code runs the function and returns the result as a "tool" message
messages.append(assistant_msg)
for call in assistant_msg.tool_calls:
    args = json.loads(call.function.arguments)  # the model's arguments arrive as a JSON string
    result = get_weather(**args)
    messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

# Step 4: the model writes the final answer using the tool result
final = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=TOOLS)
final.choices[0].message.content

'The weather in Delhi is sunny with a temperature of 30°C.'

That's ~30 lines for one tool, one provider, one round-trip. With LangChain the same thing is:
```python
@tool
def get_weather(city: str) -> str:
    "Get the current weather for a city"
    ...
agent = create_agent("gpt-4o-mini", tools=[get_weather])
agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Delhi?"}]})
```
The schema is generated from the function, the loop runs automatically, and it works with any provider.

## 3. What is an agent?

An **agent** = **LLM + tools + a loop**. The model keeps choosing actions until it decides it's done.
This pattern is called **ReAct** (*Reason + Act*):

```
        ┌──────────────────────────────────────────┐
        ▼                                          │
user ─► LLM ── wants a tool? ── yes ─► run tool ───┘   (observe the result, think again)
                    │
                    no
                    ▼
              final answer
```

### Workflow vs. agent
| | Workflow (chain / fixed graph) | Agent |
|---|---|---|
| Steps decided by | **you**, in code | **the model**, at runtime |
| Predictability | high | lower |
| Cost / latency | lower, known | variable (more LLM calls) |
| Best for | well-defined pipelines (extract → validate → save) | open-ended tasks (research, Q&A over tools) |

### Building blocks you'll add to agents
- **Memory** — short-term (the conversation in a *thread*, notebook 11); long-term (facts saved across threads).
- **Structured output** — typed final answers (notebook 12).
- **Middleware** — summarise long chats, hide PII, add logging (notebooks 13, 15).
- **Human-in-the-loop** — pause for approval before risky tools (notebook 14).

## 4. The ecosystem — which library does what?

| Library | What it is | Use it for |
|---|---|---|
| **LangChain** | standard interface to models + building blocks (messages, prompts, tools, structured output) and `create_agent` | almost everything: calling models, tools, quick agents |
| **LangGraph** | low-level **orchestration** runtime: state, nodes, edges, persistence, interrupts | custom control flow, multi-step / multi-agent systems, long-running jobs |
| **LangSmith** | tracing, debugging and evaluation platform | seeing every LLM/tool call, cost and latency (set `LANGSMITH_TRACING=true`) |

`create_agent` is built **on top of** LangGraph — so LangChain is the easy entrance and LangGraph the full control panel.
Notebook 20 is a full "when to use what" guide.

## Course roadmap
```
Part 1  Fundamentals   01 LLMs ─ 02 agents (this) ─ 03 Python prerequisites
Part 2  LangChain      04 models ─ 05 messages ─ 06 prompts ─ 07 structured output ─ 08 history
                       ─ 09 tools ─ 10 agent ─ 11 memory ─ 12 agent output ─ 13-15 middleware
Part 3  LangGraph      16 first graph ─ 17 state ─ 18 agent as graph ─ 19 research chatbot
Part 4  Guide          20 when to use what
Part 5  Projects       21 SQL agent ─ 22 dynamic tool agent
```

---
✅ **Done with 02 From LLMs to agents.** Next: [03 Python prerequisites →](03_python_prerequisites.ipynb)  
[🏠 Roadmap](../README.md)